# Conv-ViT Hybrid Framework
**Environment:** Python 3.10 | TensorFlow 2.10 | Local (non-Colab)

In [1]:
# Install compatible dependencies
# Run this cell once, then restart the kernel
import subprocess
# subprocess.run(['pip', 'install', 'tensorflow==2.10.0'], check=False)
subprocess.run(['pip', 'install', 'tensorflow-addons==0.18.0'], check=False)  # Compatible with TF 2.10
subprocess.run(['pip', 'install', 'mlxtend'], check=False)

CompletedProcess(args=['pip', 'install', 'mlxtend'], returncode=0)

In [1]:
# Import Packages
from __future__ import absolute_import, division, print_function, unicode_literals

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras.layers import (
    Input, Dense, Reshape, Flatten, Conv2D, Conv2DTranspose,
    LeakyReLU, ReLU, GlobalAveragePooling1D,
    BatchNormalization, Dropout, Embedding, Activation, Concatenate
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    TensorBoard, ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
)
# FIX: keras.utils.np_utils moved in TF 2.x
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import backend as K
K.clear_session()

import os
import numpy as np
import matplotlib.pyplot as plt
import math
import datetime
import pandas as pd
import random
from mlxtend.plotting import plot_confusion_matrix

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

TensorFlow version: 2.10.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# # Dataset Extraction (if using a zip archive)
# # Skip this cell if your dataset is already extracted
# import zipfile

# DATASET_ZIP_PATH = r""  # <-- Set your .zip file path here, e.g. r"C:/data/dataset.zip"
# EXTRACT_DIR = r""       # <-- Set extraction destination, e.g. r"C:/data/"

# if DATASET_ZIP_PATH:
#     with zipfile.ZipFile(DATASET_ZIP_PATH, 'r') as zip_ref:
#         zip_ref.extractall(EXTRACT_DIR)
#     print("Dataset extracted successfully.")

In [3]:
train_path = "E:AL/alz/Final_Journal_Simulation/Dataset/ADNI/Final_Dataset/Enhanced_Augmented_6_Class (Different Variations)/AD vs CN vs EMCI vs LMCI/"

In [4]:
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    height_shift_range=0.2,
    width_shift_range=0.2,
    zoom_range=0.2,
    rotation_range=20,
    shear_range=0.2,
    fill_mode='nearest',
    validation_split=0.2
)
test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

In [5]:
image_size = (224, 224)
batch = 4
train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=image_size,
    class_mode='categorical',
    subset = 'training',
    shuffle=True,
    batch_size=batch
)
test_data = test_datagen.flow_from_directory(
    train_path,
    target_size=image_size,
    class_mode='categorical',
    subset = 'validation',
    shuffle=False,
    batch_size=batch
)

Found 32000 images belonging to 4 classes.
Found 8000 images belonging to 4 classes.


In [6]:
IMAGE_SIZE = 224
BATCH_SIZE = 4

In [7]:
# # Data Preprocessing
# # FIX: Standardized image size to 78x78 everywhere (was inconsistent: 78 in generators, 84 in model)
# IMAGE_SIZE = 78
# BATCH_SIZE = 64

# TRAINING_DIR   = r""   # <-- e.g. r"C:/data/train"
# VALIDATION_DIR = r""   # <-- e.g. r"C:/data/val"

# training_datagen = ImageDataGenerator(rescale=1./255)
# validation_datagen = ImageDataGenerator(rescale=1./255)

# train_generator = training_datagen.flow_from_directory(
#     TRAINING_DIR,
#     target_size=(IMAGE_SIZE, IMAGE_SIZE),
#     class_mode='categorical',
#     batch_size=BATCH_SIZE
# )

# validation_generator = validation_datagen.flow_from_directory(
#     VALIDATION_DIR,
#     target_size=(IMAGE_SIZE, IMAGE_SIZE),
#     class_mode='categorical',
#     batch_size=BATCH_SIZE
# )

In [8]:
# Shared Input — used by all three sub-models
# FIX: Changed from (84,84,3) to (78,78,3) to match data generators
input_tensor = Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))

In [10]:
# Model 1: InceptionV3
pre_trained_model = tf.keras.applications.InceptionV3(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    input_tensor=input_tensor,
    include_top=False,
    weights=None
)
for layer in pre_trained_model.layers:
    layer.trainable = False

inception_layer = pre_trained_model.get_layer('mixed10').output
inception_layer = layers.Flatten()(inception_layer)
print('InceptionV3 last layer output shape:', inception_layer.shape)


# pre_trained_model.summary()

InceptionV3 last layer output shape: (None, 51200)


In [11]:
# Model 2: DenseNet121
pre_trained_model_1 = tf.keras.applications.DenseNet121(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    input_tensor=input_tensor,
    include_top=False,
    weights=None
)
for layer in pre_trained_model_1.layers:
    layer.trainable = True

densenet_layer = pre_trained_model_1.get_layer('relu').output
dense_layer = layers.Flatten()(densenet_layer)
print('DenseNet121 last layer output shape:', dense_layer.shape)


# pre_trained_model_1.summary()

DenseNet121 last layer output shape: (None, 50176)


In [27]:
# Vision Transformer (ViT) — Tunable Parameters
num_classes       = 4
patch_size        = 24
num_patches       = (IMAGE_SIZE // patch_size) ** 2
projection_dim    = 64
num_heads         = 4
transformer_units = [projection_dim * 2, projection_dim]
transformer_layers = 8
mlp_head_units    = [512, 256]

print(f"Number of patches: {num_patches}")

Number of patches: 81


In [13]:
# MLP block used inside ViT
def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x

In [14]:
# Patch extraction layer
class Patches(layers.Layer):
    def __init__(self, patch_size, **kwargs):
        super(Patches, self).__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches

    def get_config(self):
        config = super().get_config()
        config.update({'patch_size': self.patch_size})
        return config

In [15]:
# Patch encoding layer (linear projection + positional embedding)
class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super(PatchEncoder, self).__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = layers.Dense(units=projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, patch):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        encoded = self.projection(patch) + self.position_embedding(positions)
        return encoded

    def get_config(self):
        config = super().get_config()
        config.update({
            'num_patches': self.num_patches,
            'projection_dim': self.projection_dim
        })
        return config

In [28]:
# Vision Transformer branch
patches = Patches(patch_size)(input_tensor)
encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

for _ in range(transformer_layers):
    x1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    attention_output = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=projection_dim, dropout=0.1
    )(x1, x1)
    x2 = layers.Add()([attention_output, encoded_patches])
    x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
    x3 = mlp(x3, hidden_units=transformer_units, dropout_rate=0.1)
    encoded_patches = layers.Add()([x3, x2])

representation = layers.LayerNormalization(name='last_layer', epsilon=1e-6)(encoded_patches)
representation = layers.Flatten()(representation)
representation = layers.Dropout(0.5)(representation)
representation = mlp(representation, hidden_units=mlp_head_units, dropout_rate=0.5)
print('ViT last layer output shape:', representation.shape)

ViT last layer output shape: (None, 256)


In [29]:
# Merge all three branches
last_layer = layers.Concatenate()([inception_layer, dense_layer, representation])

x = layers.Dense(256, activation='relu')(last_layer)
x = layers.Dropout(0.3)(x)
x = layers.BatchNormalization(name='batch_1')(x)
x = layers.Dropout(0.2)(x)
x = ReLU()(x)

x = Dense(128)(x)
x = layers.BatchNormalization(name='batch_2')(x)
x = layers.Dropout(0.2)(x)
x = ReLU()(x)

x = Dense(64)(x)
x = BatchNormalization(name='batch_3')(x)
x = ReLU()(x)

output = layers.Dense(num_classes, activation='softmax')(x)

In [30]:
# Compile Model
model = Model(inputs=input_tensor, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=['accuracy', 'Precision', 'Recall']
)

model.summary()

Model: "model_3"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 zero_padding2d (ZeroPadding2D)  (None, 230, 230, 3)  0          ['input_1[0][0]']                
                                                                                                  
 conv1/conv (Conv2D)            (None, 112, 112, 64  9408        ['zero_padding2d[0][0]']         
                                )                                                                 
                                                                                            

In [31]:
# Callbacks
CHECKPOINT_PATH = r"best_model.h5"  # <-- Change to your desired save path
LOG_DIR = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))

callbacks = [
    ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    TensorBoard(log_dir=LOG_DIR, histogram_freq=1)
]

In [32]:
# Train Model
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=20,
    verbose=1,
    callbacks=callbacks
)

Epoch 1/20
8000/8000 [==============================] - ETA: 0s - loss: 1.4062 - accuracy: 0.2759 - precision: 0.2743 - recall: 0.0069
Epoch 1: val_accuracy improved from -inf to 0.25000, saving model to best_model.h5
8000/8000 [==============================] - 1064s 130ms/step - loss: 1.4062 - accuracy: 0.2759 - precision: 0.2743 - recall: 0.0069 - val_loss: 1.4347 - val_accuracy: 0.2500 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - lr: 0.0010
Epoch 2/20
8000/8000 [==============================] - ETA: 0s - loss: 1.3665 - accuracy: 0.2949 - precision: 0.5165 - recall: 0.0078
Epoch 2: val_accuracy did not improve from 0.25000
8000/8000 [==============================] - 1012s 127ms/step - loss: 1.3665 - accuracy: 0.2949 - precision: 0.5165 - recall: 0.0078 - val_loss: 1.3866 - val_accuracy: 0.2494 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - lr: 0.0010
Epoch 3/20
8000/8000 [==============================] - ETA: 0s - loss: 1.3499 - accuracy: 0.3160 - precision: 0.4

KeyboardInterrupt: 

In [ ]:
# Plot Training History
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

In [ ]:
# Test Dataset
TEST_DIR = r""  # <-- e.g. r"C:/data/test"

test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    class_mode='categorical',
    shuffle=False,
    batch_size=BATCH_SIZE
)

In [ ]:
# Model Evaluation
results = model.evaluate(test_generator)
print(f"\nTest Loss:      {results[0]:.4f}")
print(f"Test Accuracy:  {results[1]:.4f}")
print(f"Test Precision: {results[2]:.4f}")
print(f"Test Recall:    {results[3]:.4f}")

In [ ]:
# Confusion Matrix
import sklearn.metrics as sk_metrics

y_pred_probs = model.predict(test_generator)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

cm = sk_metrics.confusion_matrix(y_true, y_pred)
fig, ax = plot_confusion_matrix(conf_mat=cm, figsize=(8, 8), show_normed=True)
plt.title('Confusion Matrix')
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

print("\nClassification Report:")
print(sk_metrics.classification_report(
    y_true, y_pred,
    target_names=list(test_generator.class_indices.keys())
))